# Lesson 11B: Self-Organizing Maps - Practical

## Iris Species Clustering and Visualization

**Case Study**: A botanist wants to visualize relationships between iris species in a way that preserves their natural similarities. SOM creates a 2D map that reveals cluster structure.

**Goal**: Use SOM to cluster and visualize the Iris dataset.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from minisom import MiniSom
from sklearn.datasets import load_iris
from sklearn.preprocessing import MinMaxScaler

np.random.seed(42)
print('✅ Loaded!')

## Step 1: Load and Prepare Data

In [ ]:
# Load Iris dataset
iris = load_iris()
X, y = iris.data, iris.target
feature_names = iris.feature_names

print(f"Dataset shape: {X.shape}")
print(f"Features: {feature_names}")
print(f"Species: {iris.target_names}")

# SOM requires normalized inputs [0, 1]
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

print(f"\nData range after scaling:")
print(f"  Min: {X_scaled.min():.2f}")
print(f"  Max: {X_scaled.max():.2f}")
print("✅ Data prepared!")

## Step 2: Train Self-Organizing Map

**Parameters**:
- Grid: 10×10 neurons
- Sigma: Initial neighborhood radius
- Learning rate: How quickly weights adapt

In [ ]:
# Initialize and train SOM
grid_size = 10
som = MiniSom(grid_size, grid_size, X_scaled.shape[1], 
              sigma=1.5, learning_rate=0.5, random_seed=42)

# Initialize weights
som.random_weights_init(X_scaled)

# Train
n_iterations = 1000
print(f"Training SOM for {n_iterations} iterations...")
som.train_random(X_scaled, n_iterations)

print(f"✅ SOM trained on {len(X)} samples!")
print(f"   Grid: {grid_size}×{grid_size} = {grid_size**2} neurons")

## Step 3: Map Samples to SOM Grid

Each sample is assigned to its Best Matching Unit (BMU).

In [ ]:
# Get winning neurons for each sample
winner_coords = np.array([som.winner(x) for x in X_scaled])

print(f"Sample BMU assignments:")
print(f"  Shape: {winner_coords.shape}")
print(f"  Example (first 5): {winner_coords[:5]}")

# Count samples per species
for i, species in enumerate(iris.target_names):
    mask = y == i
    print(f"\n{species.capitalize()}:")
    print(f"  Samples: {mask.sum()}")
    unique_bmus = len(np.unique(winner_coords[mask], axis=0))
    print(f"  Unique BMUs: {unique_bmus}")

print("\n✅ All samples mapped!")

## Step 4: Visualize SOM Clustering

In [ ]:
# Visualize species on SOM grid
plt.figure(figsize=(12, 10))

colors = ['red', 'green', 'blue']
markers = ['o', 's', '^']

for i, species in enumerate(iris.target_names):
    mask = y == i
    coords = winner_coords[mask]
    plt.scatter(coords[:, 0], coords[:, 1], 
               label=species.capitalize(), 
               s=150, alpha=0.7, 
               c=colors[i], 
               marker=markers[i],
               edgecolors='black', linewidths=1.5)

plt.xlim(-0.5, grid_size - 0.5)
plt.ylim(-0.5, grid_size - 0.5)
plt.xlabel('SOM Grid X', fontsize=12)
plt.ylabel('SOM Grid Y', fontsize=12)
plt.title('SOM Clustering: Iris Species', fontweight='bold', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('✅ Species clustering visualized!')

## Step 5: U-Matrix Visualization

**U-Matrix** (Unified Distance Matrix) shows distances between neighboring neurons. Dark regions = cluster boundaries.

In [ ]:
# Compute U-Matrix
u_matrix = som.distance_map()

plt.figure(figsize=(10, 8))
plt.pcolor(u_matrix.T, cmap='bone_r', alpha=0.8)
plt.colorbar(label='Distance to neighbors')

# Overlay sample positions
for i, species in enumerate(iris.target_names):
    mask = y == i
    coords = winner_coords[mask]
    plt.scatter(coords[:, 0] + 0.5, coords[:, 1] + 0.5,
               label=species.capitalize(),
               s=80, alpha=0.8,
               c=colors[i],
               marker=markers[i],
               edgecolors='white', linewidths=1)

plt.xlim(0, grid_size)
plt.ylim(0, grid_size)
plt.xlabel('SOM Grid X', fontsize=12)
plt.ylabel('SOM Grid Y', fontsize=12)
plt.title('U-Matrix: Cluster Boundaries (darker = boundary)', fontweight='bold', fontsize=14)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()
print('✅ U-Matrix visualized!')

## Step 6: Feature Maps

Visualize how each feature is distributed across the SOM.

In [ ]:
# Create feature maps
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for i, feature_name in enumerate(feature_names):
    ax = axes[i]
    
    # Get weights for this feature
    weights = som.get_weights()
    feature_map = weights[:, :, i]
    
    # Plot heatmap
    im = ax.pcolor(feature_map.T, cmap='viridis', alpha=0.8)
    plt.colorbar(im, ax=ax, label=feature_name)
    
    # Overlay samples
    for j, species in enumerate(iris.target_names):
        mask = y == j
        coords = winner_coords[mask]
        ax.scatter(coords[:, 0] + 0.5, coords[:, 1] + 0.5,
                  s=30, alpha=0.5,
                  c=colors[j],
                  marker=markers[j])
    
    ax.set_xlim(0, grid_size)
    ax.set_ylim(0, grid_size)
    ax.set_xlabel('SOM X', fontsize=10)
    ax.set_ylabel('SOM Y', fontsize=10)
    ax.set_title(f'Feature Map: {feature_name}', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()
print('✅ Feature maps complete!')

## Summary

**Key Insights**:
- Setosa clearly separates from other species (bottom-left cluster)
- Versicolor and Virginica overlap slightly (top-right region)
- U-Matrix shows clear cluster boundaries
- Feature maps reveal which features drive clustering

**Production Tips**:
1. **Grid size**: Start with √(n_samples) × √(n_samples)
2. **Training**: Use 500× to 1000× number of samples for iterations
3. **Normalization**: Always scale features to [0, 1]
4. **Validation**: Check if clusters match domain knowledge

**When to use SOM**:
- Visualizing high-dimensional data
- Finding topology-preserving clusters
- Exploratory data analysis
- Quality control (manufacturing defects)